## Video 3.2.1

In [14]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

import utils

print(f"NumPy:  {np.__version__}")
print(f"PyTorch: {torch.__version__}")

NumPy:  2.5.2
PyTorch: 2.13.0+xpu


# Independent Play

This is the interactive classifier from `cool.py`, embedded directly in the notebook with Gradio.

1. Choose **blue**, **red**, or **remove**, then click the picture.
2. Choose how many steps to train and press **Train** as often as you like.
3. **Reset model** creates fresh weights but keeps your points.
4. **Clear points** removes the data.

The points are stored in `playground_points`, not inside Gradio. You can close the UI, edit `PlaygroundNet`, rerun the model cell, and launch the UI again with exactly the same data.

In [15]:
import gradio as gr

# Do not replace the list when this cell is rerun: it is our persistent dataset.
if "playground_points" not in globals():
    playground_points = []  # Each item is (x1, x2, class).

PLAYGROUND_RESOLUTION = 400
playground_axis = np.linspace(-1, 1, PLAYGROUND_RESOLUTION, dtype=np.float32)
playground_xx, playground_yy = np.meshgrid(playground_axis, playground_axis[::-1])
playground_grid = torch.from_numpy(
    np.column_stack((playground_xx.ravel(), playground_yy.ravel()))
).float()

def playground_tensors():
    if not playground_points:
        return None, None
    features = torch.tensor([point[:2] for point in playground_points], dtype=torch.float32)
    targets = torch.tensor([point[2] for point in playground_points], dtype=torch.float32)
    return features, targets

def render_playground():
    with torch.no_grad():
        probabilities = torch.sigmoid(playground_model(playground_grid).squeeze(-1))
        probabilities = probabilities.reshape(PLAYGROUND_RESOLUTION, PLAYGROUND_RESOLUTION).numpy()
    image = (plt.colormaps["coolwarm"](probabilities)[..., :3] * 255).astype(np.uint8)

    rows, columns = np.ogrid[:PLAYGROUND_RESOLUTION, :PLAYGROUND_RESOLUTION]
    for x1, x2, label in playground_points:
        center_x = round((x1 + 1) * (PLAYGROUND_RESOLUTION - 1) / 2)
        center_y = round((1 - x2) * (PLAYGROUND_RESOLUTION - 1) / 2)
        distance = (columns - center_x) ** 2 + (rows - center_y) ** 2
        image[distance <= 8 ** 2] = (255, 255, 255)
        image[distance <= 6 ** 2] = (30, 80, 230) if label == 0 else (230, 45, 45)
    return image

def playground_status(message="Ready"):
    return f"{message} | {len(playground_points)} points"

### Model definition

Edit this architecture and rerun the cell whenever you want. This resets only the model and optimizer; `playground_points` stays unchanged.

In [23]:
class PlaygroundNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 1),
            nn.ReLU(),
            nn.Linear(1, 1),
            nn.ReLU(),
            
            nn.Linear(1, 8),
            nn.ReLU(),
            
            nn.Linear(8, 1),
            nn.ReLU(),
            
            nn.Linear(1, 1),
        )

    
    def forward(self, inputs):
        return self.net(inputs).squeeze(-1)

learning_rate = 0.02


def new_playground_model():
    model = PlaygroundNet()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    return model, optimizer

playground_model, playground_optimizer = new_playground_model()
playground_loss = nn.BCEWithLogitsLoss()
print(playground_status("New model, data kept"))

New model, data kept | 3 points


In [24]:
def choose_mode(mode):
    return mode, playground_status(f"Click the image to {mode}")

def edit_point(mode, event: gr.SelectData):
    pixel_x, pixel_y = event.index
    x1 = 2 * pixel_x / (PLAYGROUND_RESOLUTION - 1) - 1
    x2 = 1 - 2 * pixel_y / (PLAYGROUND_RESOLUTION - 1)

    if mode == "remove" and playground_points:
        distances = [(px - x1) ** 2 + (py - x2) ** 2 for px, py, _ in playground_points]
        nearest = int(np.argmin(distances))
        if distances[nearest] < 0.08 ** 2:
            playground_points.pop(nearest)
    elif mode in ("blue", "red"):
        playground_points.append((x1, x2, 0 if mode == "blue" else 1))
    return render_playground(), playground_status(f"Used {mode}")

def train_playground(steps):
    features, targets = playground_tensors()
    if features is None:
        return render_playground(), playground_status("Add some points first")
    for _ in range(int(steps)):
        playground_optimizer.zero_grad()
        loss = playground_loss(playground_model(features), targets)
        loss.backward()
        playground_optimizer.step()
    return render_playground(), playground_status(f"Loss: {loss.item():.5f}")

def reset_playground_model():
    global playground_model, playground_optimizer
    playground_model, playground_optimizer = new_playground_model()
    return render_playground(), playground_status("Model reset, data kept")

def clear_playground_points():
    playground_points.clear()
    return render_playground(), playground_status("Points cleared")

with gr.Blocks(title="Neural network playground") as playground:
    gr.Markdown("## Neural network playground")
    mode = gr.State("blue")
    with gr.Row():
        with gr.Column(scale=3):
            canvas = gr.Image(render_playground(), label="Click to edit training data", interactive=False)
        with gr.Column(scale=1):
            blue_button = gr.Button("Add blue dots", variant="primary")
            red_button = gr.Button("Add red dots")
            remove_button = gr.Button("Remove dots")
            steps = gr.Slider(1, 500, value=25, step=1, label="Training steps")
            train_button = gr.Button("Train", variant="primary")
            reset_button = gr.Button("Reset model (keep dots)")
            clear_button = gr.Button("Clear dots")
            status = gr.Markdown(playground_status("Mode: blue"))

    blue_button.click(lambda: choose_mode("blue"), outputs=[mode, status])
    red_button.click(lambda: choose_mode("red"), outputs=[mode, status])
    remove_button.click(lambda: choose_mode("remove"), outputs=[mode, status])
    canvas.select(edit_point, inputs=mode, outputs=[canvas, status])
    train_button.click(train_playground, inputs=steps, outputs=[canvas, status])
    reset_button.click(reset_playground_model, outputs=[canvas, status])
    clear_button.click(clear_playground_points, outputs=[canvas, status])

playground.launch(quiet=True, prevent_thread_lock=True)

To stop the Gradio server, run the next cell. Your dots remain in `playground_points`. After editing and rerunning the model-definition cell, rerun the UI cell to continue.

In [13]:
playground.close()

Exception in callback _ProactorReadPipeTransport._loop_reading()
handle: <Handle _ProactorReadPipeTransport._loop_reading()>
Traceback (most recent call last):
  File "C:\Users\gtamir\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\gtamir\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\asyncio\proactor_events.py", line 326, in _loop_reading
    self._data_received(data, length)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "C:\Users\gtamir\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\asyncio\proactor_events.py", line 275, in _data_received
    self._protocol.data_received(data)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "C:\Users\gtamir\Intro-to-practical-AI-course\3. Deep Learning Fundementals\.venv\Lib\site-packages\uvicorn\protocols\http\h11_impl.py", line 176, in data_received
  

Closing server running on port: 7864
